# Linear Regression on HDB Transactions (2012-2021)

In [5]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

import scipy.stats as stats
from scipy.stats import zscore

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', None)

In [6]:
data = pd.read_csv("2_cleaned_data3.csv")

In [7]:
data.columns

Index(['floor_area_sqft', 'remaining_lease_years', 'region',
       'mrt_nearest_distance', 'mid', 'resale_price', 'town', 'Tranc_Year',
       'flat_model_tier', 'bto_launched', 'mop_flats'],
      dtype='str')

In [8]:
# one last check for VIF
target = "resale_price"

numeric_features = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid",
    "Tranc_Year",
    "bto_launched",
    "mop_flats"
]

categorical_features = [
    #"region",
    "town",
    "flat_model_tier"  # will be one-hot encoded
]

# build X from your dataframe
X = data[numeric_features + categorical_features]

# Preprocess: impute + one-hot encode, drop first category to avoid dummy trap
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# transform and recover feature names
X_t = preprocess.fit_transform(X)

# feature names (works for sklearn >= 1.0)
feature_names = preprocess.get_feature_names_out()

# convert to dense DataFrame for statsmodels VIF
X_df = pd.DataFrame(
    X_t.toarray() if hasattr(X_t, "toarray") else X_t,
    columns=feature_names
)

# add intercept for VIF calculation
X_df_const = sm.add_constant(X_df)

# compute VIF
vif_table = pd.DataFrame({
    "feature": X_df_const.columns,
    "VIF": [variance_inflation_factor(X_df_const.values, i) for i in range(X_df_const.shape[1])]
})

# usually we ignore const in interpretation
vif_table = vif_table.sort_values("VIF", ascending=False).reset_index(drop=True)
vif_table

,feature,VIF
0,const,1.473632e+07
1,num__Tranc_Year,2.751081e+01
2,num__mop_flats,1.913106e+01
3,num__bto_launched,4.212286e+00
4,cat__town_SENGKANG,3.070352e+00
5,cat__town_WOODLANDS,2.779193e+00
6,cat__town_JURONG WEST,2.776500e+00
7,cat__town_PUNGGOL,2.627932e+00
8,cat__town_TAMPINES,2.607185e+00
9,cat__town_YISHUN,2.393541e+00


### Observation:
- VIF values between 1-5, unlikely that data set has multicollinearity
- Low VIF value for regions imply that they are not a proxy for MRT distance
- Size (floor_area_sqft) is not rendering other features redundant

<font color = "orange">

## Model 1A: Linear Regression, raw values of data

(*subsequent models will adopt log(values) for floor_area_sqft / resale_price and add secondary features to adjust for accuracy*)

</font>

In [9]:
#redifine 1A features
numeric_features = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid",
]

categorical_features = [
    "region",
    "flat_model_tier"  # will be one-hot encoded
]

In [10]:
# define features
X = data[numeric_features + categorical_features]
y = data[target]

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=46
)

In [11]:
# preprocess for linear regression
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [12]:
# build model
model_raw = Pipeline([
    ("preprocess", preprocess),
    ("lr", LinearRegression())
])

In [13]:
# fit model
model_raw.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contain

In [14]:
# evaluate model 1A
y_pred_raw = model_raw.predict(X_test)

r2_raw = r2_score(y_test, y_pred_raw)
rmse_raw = np.sqrt(mean_squared_error(y_test, y_pred_raw))
mae_raw = mean_absolute_error(y_test, y_pred_raw)

print("RAW MODEL RESULTS")
print("R2:", r2_raw)
print("RMSE:", rmse_raw)
print("MAE:", mae_raw)

RAW MODEL RESULTS
R2: 0.7843635668546595
RMSE: 66575.51486393361
MAE: 50946.7214436157


<font color = "orange">

## Model 1B: Linear Regression using log(resale_price)

(*to adjust for accuracy*)

</font>

In [15]:
# Log transform target
y_log = np.log1p(data[target])

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=46
)

In [16]:
# build model — use SEPARATE preprocessor with Model 1B's own features
numeric_features_1b = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid",
]

categorical_features_1b = [
    "region",
    "flat_model_tier"
]

preprocess_log = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features_1b),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), categorical_features_1b)
])

model_log = Pipeline([
    ("preprocess", preprocess_log),
    ("lr", LinearRegression())
])

In [17]:
model_log.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contain

In [18]:
y_pred_log = model_log.predict(X_test)

# convert predictions back to dollar scale
y_pred_log_exp = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test_log)

r2_log = r2_score(y_test_original, y_pred_log_exp)
rmse_log = np.sqrt(mean_squared_error(y_test_original, y_pred_log_exp))
mae_log = mean_absolute_error(y_test_original, y_pred_log_exp)

print("\nLOG MODEL RESULTS (Back in $)")
print("R2:", r2_log)
print("RMSE:", rmse_log)
print("MAE:", mae_log)


LOG MODEL RESULTS (Back in $)
R2: 0.7949048875652741
RMSE: 64927.86456725196
MAE: 47543.951261760885


In [19]:
# get feature names after preprocessing
feature_names = model_log.named_steps["preprocess"].get_feature_names_out()

# get coefficients
coefficients = model_log.named_steps["lr"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df = coef_df.sort_values("coefficient", ascending=False)

coef_df

,feature,coefficient
9,cat__flat_model_tier_high_end,0.257448
10,cat__flat_model_tier_legacy,0.082307
8,cat__flat_model_tier_executive,0.055934
3,num__mid,0.009951
1,num__remaining_lease_years,0.006928
0,num__floor_area_sqft,0.000854
2,num__mrt_nearest_distance,-0.000098
11,cat__flat_model_tier_premium,-0.007675
4,cat__region_E,-0.206163
6,cat__region_NE,-0.227842


<font color = "orange">

## Model 1C: Linear Regression using log(resale_price) but removing region

(*examine how this affects R2, MAE, RMSE*)

</font>

In [20]:
X = data[numeric_features]
y_log = np.log1p(data[target])

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=46
)

In [22]:
model_no_region = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("lr", LinearRegression())
])

model_no_region.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a featu

In [23]:
y_pred_log = model_no_region.predict(X_test)

# Convert back to dollar scale
y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

r2_no_region = r2_score(y_test_original, y_pred)
rmse_no_region = np.sqrt(mean_squared_error(y_test_original, y_pred))
mae_no_region = mean_absolute_error(y_test_original, y_pred)

print("LOG MODEL WITHOUT REGION")
print("R2:", r2_no_region)
print("RMSE:", rmse_no_region)
print("MAE:", mae_no_region)

LOG MODEL WITHOUT REGION
R2: 0.5901964423051635
RMSE: 91778.57834964115
MAE: 66110.01494015745


<font color = "orange">

## Model 2A: Linear Regression using log(resale_price) but removing region

(*examine how this affects R2, MAE, RMSE*)

</font>

In [24]:
numeric_features = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid"
]

categorical_town = ["town"]

target = "resale_price"

X_base = data[numeric_features]
y_log = np.log1p(data[target])

In [25]:
X_town = data[numeric_features + categorical_town]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_town, y_log, test_size=0.2, random_state=42
)

preprocess_town = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_town)
])

In [26]:
model_town = Pipeline([
    ("preprocess", preprocess_town),
    ("lr", LinearRegression())
])

model_town.fit(X_train_t, y_train_t)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contain

In [27]:
y_pred_log_t = model_town.predict(X_test_t)

y_pred_t = np.expm1(y_pred_log_t)
y_test_original_t = np.expm1(y_test_t)

r2_town = r2_score(y_test_original_t, y_pred_t)
rmse_town = np.sqrt(mean_squared_error(y_test_original_t, y_pred_t))
mae_town = mean_absolute_error(y_test_original_t, y_pred_t)

print("\nTOWN MODEL")
print("R2:", r2_town)
print("RMSE:", rmse_town)
print("MAE:", mae_town)


TOWN MODEL
R2: 0.8639106618895307
RMSE: 52677.70533387973
MAE: 40179.897431660014


<font color = "orange">

## Model 2B: Linear Regression using log(resale_price), town instead of region, add Tranc_Year

(*examine how this affects R2, MAE, RMSE*)

</font>

In [28]:
numeric_features_year = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid",
    "Tranc_Year"
]

categorical_features = ["town"]


In [29]:
X_year = data[numeric_features_year + categorical_features]
y_log = np.log1p(data["resale_price"])

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    X_year, y_log, test_size=0.2, random_state=42
)

preprocess_year = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features_year),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

model_year = Pipeline([
    ("preprocess", preprocess_year),
    ("lr", LinearRegression())
])

model_year.fit(X_train_y, y_train_y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contain

In [30]:
y_pred_log_y = model_year.predict(X_test_y)

y_pred_y = np.expm1(y_pred_log_y)
y_test_original_y = np.expm1(y_test_y)

r2_year = r2_score(y_test_original_y, y_pred_y)
rmse_year = np.sqrt(mean_squared_error(y_test_original_y, y_pred_y))
mae_year = mean_absolute_error(y_test_original_y, y_pred_y)

print("TOWN + YEAR MODEL")
print("R2:", r2_year)
print("RMSE:", rmse_year)
print("MAE:", mae_year)

TOWN + YEAR MODEL
R2: 0.8673920090805612
RMSE: 51999.55719227454
MAE: 38045.64115006921


# Model 2C - linear regression using log(resale_price), remove tranc_year and add bto_launched and mop_flats

In [31]:
numeric_features_year = [
    "floor_area_sqft",
    "remaining_lease_years",
    "mrt_nearest_distance",
    "mid",
    "bto_launched",
    "mop_flats"
]

categorical_features = ["town"]

X_year = data[numeric_features_year + categorical_features]
y_log = np.log1p(data["resale_price"])

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    X_year, y_log, test_size=0.2, random_state=42
)

preprocess_year = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features_year),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

model_year = Pipeline([
    ("preprocess", preprocess_year),
    ("lr", LinearRegression())
])

model_year.fit(X_train_y, y_train_y)


y_pred_log_y = model_year.predict(X_test_y)

y_pred_y = np.expm1(y_pred_log_y)
y_test_original_y = np.expm1(y_test_y)

r2_year = r2_score(y_test_original_y, y_pred_y)
rmse_year = np.sqrt(mean_squared_error(y_test_original_y, y_pred_y))
mae_year = mean_absolute_error(y_test_original_y, y_pred_y)

print("TOWN + BTO/MOP model")
print("R2:", r2_year)
print("RMSE:", rmse_year)
print("MAE:", mae_year)

TOWN + BTO/MOP model
R2: 0.8717610389012049
RMSE: 51135.769210937906
MAE: 36777.50570433512


## Export model to .plk

In [32]:
import joblib

In [33]:
print(type(model_log))

<class 'sklearn.pipeline.Pipeline'>


In [34]:
model_log.predict(X_test.head())

ValueError: columns are missing: {'flat_model_tier', 'region'}

In [ ]:
joblib.dump(model_log, "hdb_lr_log_pipeline.pkl")

['hdb_lr_log_pipeline.pkl']

This notebook builds a baseline linear regression model to predict resale prices, applying log transformation to address skewness, evaluating performance using RMSE and R², and analysing residuals to validate modelling assumptions. The model provides interpretable coefficients that quantify the marginal impact of property features on price, serving as a benchmark for more advanced predictive models such as RandomForest and XGboost